<a href="https://colab.research.google.com/github/subhajit404/RNN/blob/main/Rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('qoute_dataset.csv')

In [3]:
quotes = df['quote']

In [4]:
quotes = quotes.str.lower()

In [5]:
quotes = quotes.str.replace(r'[^\w\s]', '', regex=True)

In [6]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer

In [7]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [8]:
word_idx = tokenizer.word_index
print(len(word_idx))
list(word_idx.items())[:10]

8723


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [9]:
sequence = tokenizer.texts_to_sequences(quotes)

In [10]:
for i in range (3):
    print(quotes[i])

the world as we have created it is a process of our thinking it cannot be changed without changing our thinking
it is our choices harry that show what we truly are far more than our abilities
there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle


In [11]:
X = []
y = []

In [12]:
for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [13]:
max_len= max(len(x) for x in X)
max_len

745

In [14]:
import tensorflow
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [15]:
X_pad = pad_sequences(X,maxlen=max_len,padding='pre')

In [16]:
y= np.array(y)

In [17]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes= vocab_size)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [19]:
emb_dim = 50
rnn_units = 128

In [20]:
rnn_model =Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=emb_dim,input_length=max_len)
)

rnn_model.add(
    SimpleRNN(units=rnn_units)
)
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

In [21]:
rnn_model.compile(
    optimizer='adam',
    loss= 'categorical_crossentropy',
    metrics = ['accuracy']
    )

In [22]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [23]:
rnn_model.fit(X_pad,y_one_hot ,epochs=100,batch_size=64,verbose=1)

Epoch 1/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.0517 - loss: 6.6148
Epoch 2/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.0927 - loss: 5.9345
Epoch 3/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.1144 - loss: 5.5517
Epoch 4/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 81s 57ms/step - accuracy: 0.1310 - loss: 5.2215
Epoch 5/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 76s 57ms/step - accuracy: 0.1478 - loss: 4.9178
Epoch 6/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 76s 57ms/step - accuracy: 0.1664 - loss: 4.6314
Epoch 7/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 76s 57ms/step - accuracy: 0.1908 - loss: 4.3751
Epoch 8/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 76s 57ms/step - accuracy: 0.2123 - loss: 4.1901
Epoch 9/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.2450 - loss: 3.9170
Epoch 10/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.2718 - loss: 3.7336
Epoch 11/100
1332/1332 ━━━━━━━━━━━━━━━━━━━━ 77s 58ms/step - accuracy: 0.3009 - loss: 3.54

In [25]:
rnn_model.save("Rnn.h5")